# Departments Exploratory Data Analysis

## Purpose

Check whether department records are complete, easy to understand and safe to connect to organisations, branches and wellness data.

## Files used

- `Departments.csv` — department details
- `Organisations.csv` — organisations that own the departments
- `Branches.csv` — organisation locations

There are four departments from one organisation. The data can show whether the current records work, but it cannot show patterns across many clients.

## 1. Set up the analysis

Import pandas and find the raw data folder without showing anyone's personal computer path.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (
            directory / "data" / "raw",
            directory / "data-analytics" / "data" / "raw",
        ):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate the repository raw-data folder.")

RAW_DATA_DIR = find_raw_data_dir()

## 2. Load the data

Load departments, organisations and branches. Show the number of rows and columns in each file.

In [2]:
departments = pd.read_csv(RAW_DATA_DIR / "Departments.csv")
organisations = pd.read_csv(RAW_DATA_DIR / "Organisations.csv")
branches = pd.read_csv(RAW_DATA_DIR / "Branches.csv")

dataset_summary = pd.DataFrame({
    "dataset": ["Departments", "Organisations", "Branches"],
    "rows": [len(departments), len(organisations), len(branches)],
    "columns": [
        len(departments.columns),
        len(organisations.columns),
        len(branches.columns),
    ],
})

dataset_summary

,dataset,rows,columns
0,Departments,4,4
1,Organisations,1,10
2,Branches,2,6


## 3. View the department records

Display the department IDs, names, organisation IDs and creation dates.

In [3]:
departments

,department_id,organisation_id,name,created_at
0,DEP-001,ORG-001,Operations,2026-05-15T09:10:00Z
1,DEP-002,ORG-001,Engineering,2026-05-15T09:11:00Z
2,DEP-003,ORG-001,Finance,2026-05-15T09:12:00Z
3,DEP-004,ORG-001,Human Resources,2026-05-15T09:13:00Z


## 4. Check the columns

Confirm that the fields needed for an organisation-level department record are available. Also check whether departments have a `branch_id`.

In [4]:
required_columns = {
    "department_id",
    "organisation_id",
    "name",
    "created_at",
}

missing_required_columns = sorted(required_columns - set(departments.columns))

schema_summary = pd.Series({
    "required_columns_present": len(missing_required_columns) == 0,
    "department_id_available": "department_id" in departments.columns,
    "organisation_id_available": "organisation_id" in departments.columns,
    "branch_id_available": "branch_id" in departments.columns,
})

print("Missing required columns:", missing_required_columns)
schema_summary.to_frame("result")

Missing required columns: []


,result
required_columns_present,True
department_id_available,True
organisation_id_available,True
branch_id_available,False


### What the columns mean

The current fields can connect each department to an organisation. A missing `branch_id` means Pulse80 cannot tell which branch contains the department.

## 5. Check missing values and duplicates

Show the data type, missing values and unique values for each column. Also count rows that are completely repeated.

In [5]:
department_profile = pd.DataFrame({
    "data_type": departments.dtypes.astype(str),
    "missing_count": departments.isna().sum(),
    "unique_values": departments.nunique(dropna=False),
})

print("Fully duplicated rows:", departments.duplicated().sum())
department_profile

Fully duplicated rows: 0


,data_type,missing_count,unique_values
department_id,object,0,4
organisation_id,object,0,1
name,object,0,4
created_at,object,0,4


## 6. Check department IDs and names

Check that every ID exists, is unique and follows the `DEP-001` format. Also look for repeated department names inside the same organisation.

In [6]:
normalised_department_names = (
    departments["name"].astype("string").str.strip().str.casefold()
)

identifier_checks = pd.Series({
    "missing_department_ids": departments["department_id"].isna().sum(),
    "duplicate_department_ids": departments["department_id"].duplicated().sum(),
    "invalid_department_id_formats": (
        ~departments["department_id"]
        .astype("string")
        .str.match(r"^DEP-[0-9]{3,}$", na=False)
    ).sum(),
    "duplicate_names_within_organisation": (
        departments.assign(_normalised_name=normalised_department_names)
        .duplicated(subset=["organisation_id", "_normalised_name"])
        .sum()
    ),
})

identifier_checks.to_frame("count")

,count
missing_department_ids,0
duplicate_department_ids,0
invalid_department_id_formats,0
duplicate_names_within_organisation,0


### Confirm the ID checks

Stop the analysis if an ID or department-name check fails.

In [7]:
assert identifier_checks.eq(0).all()
print("Department identifier and name checks passed.")

Department identifier and name checks passed.


## 7. Check required values and dates

Look for missing or blank IDs and names. Convert `created_at` into a date and count dates that cannot be read.

In [8]:
required_text_columns = ["department_id", "organisation_id", "name"]

required_text_checks = pd.Series({
    column: (
        departments[column].isna()
        | departments[column]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
    ).sum()
    for column in required_text_columns
})

departments["created_at"] = pd.to_datetime(
    departments["created_at"],
    utc=True,
    errors="coerce",
)

value_checks = pd.concat([
    required_text_checks.rename(lambda name: f"invalid_{name}"),
    pd.Series({
        "invalid_created_at_dates": departments["created_at"].isna().sum()
    }),
])

value_checks.to_frame("count")

,count
invalid_department_id,0
invalid_organisation_id,0
invalid_name,0
invalid_created_at_dates,0


### Confirm the required values

Stop the analysis if an important value is missing or a creation date is invalid.

In [9]:
assert value_checks.eq(0).all()
print("Required department value checks passed.")

Required department value checks passed.


## 8. Count departments per organisation

Show how many departments each organisation has and list their names.

In [10]:
departments_per_organisation = (
    departments.groupby("organisation_id", dropna=False)
    .agg(
        department_count=("department_id", "nunique"),
        department_names=("name", lambda values: sorted(values.tolist())),
    )
    .reset_index()
    .sort_values("department_count", ascending=False)
)

departments_per_organisation

,organisation_id,department_count,department_names
0,ORG-001,4,"[Engineering, Finance, Human Resources, Operations]"


## 9. Review department names

Count how often each department name appears. This can reveal repeated names and spelling differences when more organisations are added.

In [11]:
department_name_distribution = (
    departments.assign(
        normalised_name=departments["name"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )
    .groupby("normalised_name", dropna=False)
    .agg(
        record_count=("department_id", "size"),
        organisation_count=("organisation_id", "nunique"),
    )
    .reset_index()
    .sort_values(["record_count", "normalised_name"], ascending=[False, True])
)

department_name_distribution

,normalised_name,record_count,organisation_count
0,engineering,1,1
1,finance,1,1
2,human resources,1,1
3,operations,1,1


## 10. Connect departments to organisations

Join each department to its organisation. Check that every organisation ID exists and that no department rows are lost.

In [12]:
department_organisation_join = departments.merge(
    organisations[["organisation_id", "name"]].rename(
        columns={"name": "organisation_name"}
    ),
    on="organisation_id",
    how="left",
    validate="many_to_one",
    indicator="organisation_join_status",
)

department_organisation_join

,department_id,organisation_id,name,created_at,organisation_name,organisation_join_status
0,DEP-001,ORG-001,Operations,2026-05-15 09:10:00+00:00,Kopano Mining Group,both
1,DEP-002,ORG-001,Engineering,2026-05-15 09:11:00+00:00,Kopano Mining Group,both
2,DEP-003,ORG-001,Finance,2026-05-15 09:12:00+00:00,Kopano Mining Group,both
3,DEP-004,ORG-001,Human Resources,2026-05-15 09:13:00+00:00,Kopano Mining Group,both


### Confirm the organisation connection

Stop the analysis if a department cannot find its organisation or if the join changes the number of department rows.

In [13]:
assert department_organisation_join[
    "organisation_join_status"
].eq("both").all()
assert len(department_organisation_join) == len(departments)
print("All departments join to their organisations correctly.")

All departments join to their organisations correctly.


## 11. Check the branch relationship

Departments and branches both contain `organisation_id`, but departments do not contain `branch_id`. Count the possible branches for each department.

In [14]:
department_branch_candidates = departments.merge(
    branches[["branch_id", "organisation_id", "name"]].rename(
        columns={"name": "branch_name"}
    ),
    on="organisation_id",
    how="left",
    validate="many_to_many",
)

branch_candidates_per_department = (
    department_branch_candidates.groupby(
        ["department_id", "name"],
        dropna=False,
    )
    .agg(
        possible_branch_count=("branch_id", "nunique"),
        possible_branches=("branch_name", lambda values: sorted(values.dropna().tolist())),
    )
    .reset_index()
)

branch_candidates_per_department

,department_id,name,possible_branch_count,possible_branches
0,DEP-001,Operations,2,"[Gaborone Head Office, Metsi Operations Site]"
1,DEP-002,Engineering,2,"[Gaborone Head Office, Metsi Operations Site]"
2,DEP-003,Finance,2,"[Gaborone Head Office, Metsi Operations Site]"
3,DEP-004,Human Resources,2,"[Gaborone Head Office, Metsi Operations Site]"


### What the branch result means

Each department matches both branches because the join only uses `organisation_id`. Pulse80 cannot tell which branch is correct. Joining this way would repeat department rows and could place one department under several branches.

## 12. Show the final results

Separate clean-record checks from missing relationships. Clean records can still be unsuitable for branch-level reporting.

In [15]:
data_quality_summary = pd.Series({
    "fully_duplicated_rows": departments.duplicated().sum(),
    "missing_department_ids": departments["department_id"].isna().sum(),
    "duplicate_department_ids": departments["department_id"].duplicated().sum(),
    "invalid_department_id_formats": identifier_checks[
        "invalid_department_id_formats"
    ],
    "duplicate_names_within_organisation": identifier_checks[
        "duplicate_names_within_organisation"
    ],
    "invalid_required_values": required_text_checks.sum(),
    "invalid_created_at_dates": departments["created_at"].isna().sum(),
    "invalid_organisation_references": (
        ~department_organisation_join[
            "organisation_join_status"
        ].eq("both")
    ).sum(),
    "department_rows_lost_during_join": (
        len(departments) - len(department_organisation_join)
    ),
})

structural_readiness = pd.Series({
    "branch_id_missing_from_schema": int(
        "branch_id" not in departments.columns
    ),
    "departments_with_multiple_possible_branches": (
        branch_candidates_per_department["possible_branch_count"] > 1
    ).sum(),
})

print("Record quality:")
display(data_quality_summary.to_frame("failed_records"))

print("Structural readiness:")
display(structural_readiness.to_frame("affected_items"))

Record quality:
                                     failed_records
fully_duplicated_rows                             0
missing_department_ids                            0
duplicate_department_ids                          0
invalid_department_id_formats                     0
duplicate_names_within_organisation               0
invalid_required_values                           0
invalid_created_at_dates                          0
invalid_organisation_references                   0
department_rows_lost_during_join                  0
Structural readiness:
                                             affected_items
branch_id_missing_from_schema                             1
departments_with_multiple_possible_branches               4


### Confirm the data quality

Confirm that the current department records are clean at organisation level. The missing branch relationship needs a data-model change, not a simple data-cleaning fix.

In [16]:
assert data_quality_summary.eq(0).all()
print("Department records pass organisation-level quality checks.")

Department records pass organisation-level quality checks.


## 13. Findings and recommendations

The four department records are complete and consistent.

- Every department has a unique ID in the correct format.
- Every department has a name and a valid creation date.
- All departments connect to an organisation that exists.
- There are no repeated department records or repeated names.
- The organisation has Operations, Engineering, Finance and Human Resources departments.

The department IDs are suitable for connecting departments to organisations.

However, `Departments.csv` does not have `branch_id`. The organisation has two branches, so every department currently matches both branches. Pulse80 cannot tell which branch a department belongs to.

The data can support department lists and counts for an organisation. It cannot safely support department information for a specific branch.

Recommended changes:

- Add `branch_id` to each department record.
- Check that every `branch_id` exists in `Branches.csv`.
- Check that the branch and department belong to the same organisation.
- Store `department_id` in screening and participation records instead of relying on typed department names.
- Repeat the analysis when more organisations and departments are available.

**Overall result:** `Departments.csv` is ready for organisation-level use. A branch relationship is needed before Pulse80 can use it for branch-level reports and comparisons.